## EDA y limpieza de datos

El Análisis Exploratorio de Datos (EDA) permite conocer la estructura, calidad y características iniciales de los datos antes de utilizarlos en análisis posteriores. En este notebook se revisan los conjuntos de inventario, compras y ventas; después se realizan limpiezas puntuales para mejorar su consistencia. Los archivos originales se conservan y los resultados se guardan por separado. El alcance actual se centra en la estructura y calidad de los datos; no incluye todavía análisis de tendencias ni visualizaciones.

En cada etapa, las vistas previas ayudan a verificar la lectura; la revisión de tipos y valores faltantes identifica problemas de calidad; la estandarización y limpieza facilitan comparaciones y agrupaciones confiables.

## **LIMPIEZA DE DATASET INVENTARIO**

1. **Carga y vista previa:** se lee la hoja `CONSULTAPRODUCTOSMODELOIA` y se muestran sus primeras filas para verificar que el archivo y los campos se importaron correctamente. También se genera una copia CSV para disponer de los datos fuera de Excel.
2. **Estructura y esquema:** `info()` y `columns` muestran el tamaño, los tipos de datos y los nombres originales. Esta revisión permite anticipar conversiones o problemas antes de analizar.
3. **Nombres estandarizados:** se renombran las columnas con términos más claros (`codigo`, `descripcion`, `stock`, etc.) para facilitar su interpretación y uso posterior.
4. **Calidad:** se cuentan nulos, filas duplicadas y textos con espacios al inicio o al final. Estas comprobaciones detectan registros incompletos, repetidos o inconsistentes.
5. **Limpieza y comprobación:** se quitan espacios externos de la descripción y categoría, se guarda `data/processed/INVENTARIO_PRODUCTOS_LIMPIO.csv` y se vuelve a cargar para comprobar el resultado y visualizar algunos productos.

In [1]:
import pandas as pd 
from pathlib import Path

root = Path.cwd()
while not (root / "data" / "raw").is_dir():
    if root.parent == root:
        raise FileNotFoundError("No se encontró la carpeta data/raw del proyecto.")
    root = root.parent
raw_dir = root / "data" / "raw"
processed_dir = root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df_inventario = pd.read_excel(raw_dir / 'INVENTARIOPRODUCTOS.xlsx', sheet_name='CONSULTAPRODUCTOSMODELOIA')

df_inventario.to_csv(processed_dir / 'INVENTARIO_PRODUCTOS.csv', index=False)

df_inventario.head()


,REFE,NOMBRE,COSTO,PRECIO,SALDOREAL,GRU_DES
0,15,LLANTA GOLD PARTINER 12R 22.5 GP717A TRA,829411.7600,1.316000e+06,0,LLANTAS
1,35,LLANTA PIRELLI 750-16,331095.8571,5.253388e+05,0,LLANTAS
2,46,LLANTA CONTINENTAL 235-55-20 CROSSCONTACT UHP,730900.0000,1.159695e+06,0,LLANTAS
3,48,LLANTA APTANY 175-70-14C 8 LONAS 95/93T RL106,179900.0000,2.854413e+05,0,LLANTAS
4,49,LLANTA MOTO 110-80-18 MR923,270000.0000,4.284000e+05,0,LLANTAS


In [2]:
df_inventario.info()


<class 'pandas.DataFrame'>
RangeIndex: 2159 entries, 0 to 2158
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   REFE       2159 non-null   int64  
 1   NOMBRE     2159 non-null   str    
 2   COSTO      2159 non-null   float64
 3   PRECIO     2159 non-null   float64
 4   SALDOREAL  2159 non-null   int64  
 5   GRU_DES    2159 non-null   str    
dtypes: float64(2), int64(2), str(2)
memory usage: 101.3 KB


In [3]:
df_inventario.columns

Index(['REFE', 'NOMBRE', 'COSTO', 'PRECIO', 'SALDOREAL', 'GRU_DES'], dtype='str')

In [4]:
renom_colum = {
    'REFE': 'codigo', 
    'NOMBRE': 'descripcion',
    'COSTO': 'costo_c(COP)',
    'PRECIO': 'precio_v(COP)', 
    'SALDOREAL': 'stock', 
    'GRU_DES': 'categoria'
}

pd.set_option('display.float_format', lambda X: '%.2F' % X)

df_inventario = df_inventario.rename(columns=renom_colum)
df_inventario.head()

,codigo,descripcion,costo_c(COP),precio_v(COP),stock,categoria
0,15,LLANTA GOLD PARTINER 12R 22.5 GP717A TRA,829411.76,1315999.99,0,LLANTAS
1,35,LLANTA PIRELLI 750-16,331095.86,525338.76,0,LLANTAS
2,46,LLANTA CONTINENTAL 235-55-20 CROSSCONTACT UHP,730900.00,1159694.67,0,LLANTAS
3,48,LLANTA APTANY 175-70-14C 8 LONAS 95/93T RL106,179900.00,285441.33,0,LLANTAS
4,49,LLANTA MOTO 110-80-18 MR923,270000.00,428400.00,0,LLANTAS


In [5]:
nulos = df_inventario.isna().sum()

duplicados = df_inventario.duplicated().sum()

espacios = df_inventario.select_dtypes(include='object').apply(
    lambda col: col.astype(str).str.contains(r'^\s+|\s+$', regex=True).sum()
)

print(f" ¿Registros totalmente duplicados: {duplicados}")
print("\n Valores nulos por columna:")
print(nulos)
print("\n Filas con espacios en blanco extras por columna:")
print(espacios)

 ¿Registros totalmente duplicados: 0

 Valores nulos por columna:
codigo           0
descripcion      0
costo_c(COP)     0
precio_v(COP)    0
stock            0
categoria        0
dtype: int64

 Filas con espacios en blanco extras por columna:
descripcion    173
categoria        0
dtype: int64


C:\Users\aleja\AppData\Local\Temp\ipykernel_21780\927775175.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  espacios = df_inventario.select_dtypes(include='object').apply(


In [6]:
df_inventario['descripcion'] = df_inventario['descripcion'].astype(str).str.strip()
df_inventario['categoria'] = df_inventario['categoria'].astype(str).str.strip()

espacios_act = df_inventario.select_dtypes(include='object').apply(
    lambda col: col.astype(str).str.contains(r'^\s+|\s+$', regex=True).sum()
)
print("\n Filas con espacios en blanco extras por columna:")
print(espacios_act)

df_inventario.to_csv(processed_dir / 'INVENTARIO_PRODUCTOS_LIMPIO.csv', index=False)
print("✅ Archivo 'data/processed/INVENTARIO_PRODUCTOS_LIMPIO.csv' guardado correctamente.")


 Filas con espacios en blanco extras por columna:
descripcion    0
categoria      0
dtype: int64
✅ Archivo 'data/processed/INVENTARIO_PRODUCTOS_LIMPIO.csv' guardado correctamente.


C:\Users\aleja\AppData\Local\Temp\ipykernel_21780\2838700879.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  espacios_act = df_inventario.select_dtypes(include='object').apply(


In [7]:
import pandas as pd

df_inventario_limpio = pd.read_csv(processed_dir / 'INVENTARIO_PRODUCTOS_LIMPIO.csv')

pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 3. Mostrar las primeras 5 filas del inventario listo
print(f"✅ Inventario cargado con éxito. Total de productos: {len(df_inventario_limpio)}")
display(df_inventario_limpio.head())

✅ Inventario cargado con éxito. Total de productos: 2159


,codigo,descripcion,costo_c(COP),precio_v(COP),stock,categoria
0,15,LLANTA GOLD PARTINER 12R 22.5 GP717A TRA,829411.76,1315999.99,0,LLANTAS
1,35,LLANTA PIRELLI 750-16,331095.86,525338.76,0,LLANTAS
2,46,LLANTA CONTINENTAL 235-55-20 CROSSCONTACT UHP,730900.00,1159694.67,0,LLANTAS
3,48,LLANTA APTANY 175-70-14C 8 LONAS 95/93T RL106,179900.00,285441.33,0,LLANTAS
4,49,LLANTA MOTO 110-80-18 MR923,270000.00,428400.00,0,LLANTAS


## **LIMPIEZA DE DATASET COMPRAS**

1. **Carga y vista previa:** se importa la hoja `COMPRAS` y se muestran filas de ejemplo. La copia `data/processed/DS_COMPRAS.csv` permite conservar una salida de trabajo sin sobrescribir el Excel original.
2. **Estructura:** `info()` permite revisar el volumen de registros, los tipos de datos y los campos con valores faltantes antes de transformar la tabla.
3. **Nombres y fecha:** se renombran los campos para que sean más comprensibles y se normaliza `fecha` para mantener una fecha consistente, sin diferencias causadas por la hora.
4. **Calidad:** se cuentan nulos, duplicados y espacios externos en los campos de texto. Esta auditoría ayuda a determinar si la tabla necesita limpieza adicional.
5. **Preparación de ventas:** la celda siguiente exporta también una copia de la hoja `VENTAS` (`data/processed/DS_VENTAS.csv`) para su revisión posterior. Se usa una variable separada para no reemplazar por error la tabla de compras.

In [8]:
df_compras = pd.read_excel(raw_dir / 'ANALISISVENTASYCOMPRASIA.xlsx', sheet_name='COMPRAS')

df_compras.to_csv(processed_dir / 'DS_COMPRAS.csv', index=False)

df_compras.head()


,FECHA,FACTURA,ENTRADA,NIT,NOMBRE,REFE,DESCRIPCION,KAR_ENT,KAR_PRE,DOC_OBS
0,2025-02-11,N0093,112861,70115272,VARGAS ENRIQUE,6811,EXTRACCION DE RODAMIENTOS,1,55000.00,OT 3373600 OML602 MPIO MEDELLIN NB
1,2025-02-13,E12986,112954,800137876,INTERMOVILES LIMITADA,8562,RESORTES VARIOS,1,98000.00,REF 3881 (2) OT 346170 SERAUTOS / REF 8562 (1...
2,2025-02-14,NRBC12650,112958,800095007,IMPORTADORA NIPON S.A,9913,VALVULA SELLOMATIC,1,1465672.00,OT 498115 ZXZ143 DIEGO VELASQUEZ NB
3,2025-02-14,FEV108332,112970,811006765,UCAYALI S.A.S.,1030,FILTRO AIRE HYUNDAI TUCSON IX35 49210/AP3145/A...,6,15361.00,STOCK PH
4,2025-02-14,FEV108332,112970,811006765,UCAYALI S.A.S.,8641,FILTRO AIRE SKODA YETI / CROSSFOX AIP1044 FAP...,5,14812.30,STOCK PH


In [9]:
df_compras.info()

<class 'pandas.DataFrame'>
RangeIndex: 31131 entries, 0 to 31130
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   FECHA        31131 non-null  datetime64[us]
 1   FACTURA      31131 non-null  str           
 2   ENTRADA      31131 non-null  int64         
 3   NIT          31131 non-null  int64         
 4   NOMBRE       31131 non-null  str           
 5   REFE         31131 non-null  int64         
 6   DESCRIPCION  31131 non-null  str           
 7   KAR_ENT      31131 non-null  int64         
 8   KAR_PRE      31131 non-null  float64       
 9   DOC_OBS      31121 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), str(4)
memory usage: 2.4 MB


In [10]:
renom_compras = {
    'FECHA': 'fecha',
    'FACTURA': 'num_factura',
    'ENTRADA': 'num_entrada',
    'NIT': 'nit_proveedor',
    'NOMBRE': 'nom_proveedor',
    'REFE': 'codigo',
    'DESCRIPCION': 'descripción',
    'KAR_ENT': 'cantidad_comprada',
    'KAR_PRE': 'costo_unitario',
    'DOC_OBS': 'observacion'
}

df_compras = df_compras.rename(columns=renom_compras)

pd.set_option('display.float_format', lambda x: '%.2f' % x)

df_compras['fecha'] = pd.to_datetime(df_compras['fecha']).dt.normalize()
df_compras.head()


,fecha,num_factura,num_entrada,nit_proveedor,nom_proveedor,codigo,descripción,cantidad_comprada,costo_unitario,observacion
0,2025-02-11,N0093,112861,70115272,VARGAS ENRIQUE,6811,EXTRACCION DE RODAMIENTOS,1,55000.00,OT 3373600 OML602 MPIO MEDELLIN NB
1,2025-02-13,E12986,112954,800137876,INTERMOVILES LIMITADA,8562,RESORTES VARIOS,1,98000.00,REF 3881 (2) OT 346170 SERAUTOS / REF 8562 (1...
2,2025-02-14,NRBC12650,112958,800095007,IMPORTADORA NIPON S.A,9913,VALVULA SELLOMATIC,1,1465672.00,OT 498115 ZXZ143 DIEGO VELASQUEZ NB
3,2025-02-14,FEV108332,112970,811006765,UCAYALI S.A.S.,1030,FILTRO AIRE HYUNDAI TUCSON IX35 49210/AP3145/A...,6,15361.00,STOCK PH
4,2025-02-14,FEV108332,112970,811006765,UCAYALI S.A.S.,8641,FILTRO AIRE SKODA YETI / CROSSFOX AIP1044 FAP...,5,14812.30,STOCK PH


In [11]:
nulos_1 = df_compras.isna().sum()

duplicados_1 = df_compras.duplicated().sum()

espacios_1 = df_compras.select_dtypes(include='object').apply(
    lambda col: col.astype(str).str.contains(r'^\s+|\s+$', regex=True).sum()
)

print(f"📊 Registros totalmente duplicados: {duplicados_1}")
print("\n Valores nulos por columna:")
print(nulos_1)
print("\n Filas con espacios en blanco extras por columna:")
print(espacios_1)

📊 Registros totalmente duplicados: 208

 Valores nulos por columna:
fecha                 0
num_factura           0
num_entrada           0
nit_proveedor         0
nom_proveedor         0
codigo                0
descripción           0
cantidad_comprada     0
costo_unitario        0
observacion          10
dtype: int64

 Filas con espacios en blanco extras por columna:
num_factura         0
nom_proveedor       0
descripción      7369
observacion        48
dtype: int64


C:\Users\aleja\AppData\Local\Temp\ipykernel_21780\3396390299.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  espacios_1 = df_compras.select_dtypes(include='object').apply(


In [12]:
df_ventas_exportacion = pd.read_excel(raw_dir / 'ANALISISVENTASYCOMPRASIA.xlsx', sheet_name='VENTAS')

df_ventas_exportacion.to_csv(processed_dir / 'DS_VENTAS.csv', index=False)

df_ventas_exportacion.head()


,FAC_NUM,FAC_FEC,FAC_VEH,KAR_COD,REF_DES,KAR_SAL,KAR_PRE,KAR_IVA,KAR_VIS,SUBTOTAL,VRIVA,VRTOTAL,GRU_DES
0,24145,2025-01-02,NaN,729,ACEITE MOBIL 20W50 4T,1.00,26000.00,0,0.00,26000.00,0.00,26000.00,ACEITES
1,24146,2025-01-02,NaN,4238,AMBIENTADOR LIGHT GEL,1.00,11764.70,19,2235.29,11764.70,2235.29,13999.99,ACCESORIOS
2,24147,2025-01-02,HYT526,517,MONTAJE DE LLANTAS,1.00,12605.00,19,2394.95,12605.00,2394.95,14999.95,SERVICIOS
3,24148,2025-01-02,LTS797,542,LAVADO EXTERIOR,1.00,27731.00,19,5268.89,27731.00,5268.89,32999.89,SERVICIOS
4,24148,2025-01-02,LTS797,547,LAVADO DE CHASIS,1.00,43697.00,19,8302.43,43697.00,8302.43,51999.43,SERVICIOS


## Limpieza de dataset ventas

1. **Carga y respaldo:** se lee la hoja `VENTAS` y se conserva una copia original (`df_ventas_original`) para comparar los datos antes y después sin alterar la fuente. La vista previa verifica la importación.
2. **Diagnóstico:** `info()` y `columns` muestran el tamaño, los tipos y los campos. Luego se miden nulos, duplicados y espacios externos antes de limpiar; esto establece una línea base de calidad.
3. **Estandarización y limpieza:** se renombran las columnas; se recortan espacios en descripción, categoría y vehículo; las placas faltantes se marcan como `SIN_PLACA` para conservar esas ventas; y se eliminan filas totalmente duplicadas.
4. **Resultado y verificación:** se guarda el CSV limpio y se comparan métricas antes/después. La comparación permite confirmar el efecto de la limpieza y comprobar que las ventas sin placa siguen presentes.

In [13]:
import pandas as pd
from pathlib import Path

archivo_ventas = root / "data" / "raw" / "ANALISISVENTASYCOMPRASIA.xlsx"

if not archivo_ventas.exists():
    raise FileNotFoundError(f"No se encontr? el archivo de ventas: {archivo_ventas}")

df_ventas_original = pd.read_excel(archivo_ventas, sheet_name="VENTAS")
df_ventas = df_ventas_original.copy()
df_ventas.head()

,FAC_NUM,FAC_FEC,FAC_VEH,KAR_COD,REF_DES,KAR_SAL,KAR_PRE,KAR_IVA,KAR_VIS,SUBTOTAL,VRIVA,VRTOTAL,GRU_DES
0,24145,2025-01-02,NaN,729,ACEITE MOBIL 20W50 4T,1.00,26000.00,0,0.00,26000.00,0.00,26000.00,ACEITES
1,24146,2025-01-02,NaN,4238,AMBIENTADOR LIGHT GEL,1.00,11764.70,19,2235.29,11764.70,2235.29,13999.99,ACCESORIOS
2,24147,2025-01-02,HYT526,517,MONTAJE DE LLANTAS,1.00,12605.00,19,2394.95,12605.00,2394.95,14999.95,SERVICIOS
3,24148,2025-01-02,LTS797,542,LAVADO EXTERIOR,1.00,27731.00,19,5268.89,27731.00,5268.89,32999.89,SERVICIOS
4,24148,2025-01-02,LTS797,547,LAVADO DE CHASIS,1.00,43697.00,19,8302.43,43697.00,8302.43,51999.43,SERVICIOS


In [14]:
df_ventas.info()
print("Columnas:")
print(df_ventas.columns)

<class 'pandas.DataFrame'>
RangeIndex: 128086 entries, 0 to 128085
Data columns (total 13 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   FAC_NUM   128086 non-null  int64         
 1   FAC_FEC   128086 non-null  datetime64[us]
 2   FAC_VEH   99073 non-null   str           
 3   KAR_COD   128086 non-null  int64         
 4   REF_DES   128086 non-null  str           
 5   KAR_SAL   128086 non-null  float64       
 6   KAR_PRE   128086 non-null  float64       
 7   KAR_IVA   128086 non-null  int64         
 8   KAR_VIS   128086 non-null  float64       
 9   SUBTOTAL  128086 non-null  float64       
 10  VRIVA     128086 non-null  float64       
 11  VRTOTAL   128086 non-null  float64       
 12  GRU_DES   128086 non-null  str           
dtypes: datetime64[us](1), float64(6), int64(3), str(3)
memory usage: 12.7 MB
Columnas:
Index(['FAC_NUM', 'FAC_FEC', 'FAC_VEH', 'KAR_COD', 'REF_DES', 'KAR_SAL',
       'KAR_PRE', 'KAR_IVA', 'KAR_

In [15]:
renombrar_ventas = {
    "FAC_NUM": "factura",
    "FAC_FEC": "fecha",
    "FAC_VEH": "vehiculo",
    "KAR_COD": "codigo",
    "REF_DES": "descripcion",
    "KAR_SAL": "cantidad",
    "KAR_PRE": "precio_unitario",
    "KAR_IVA": "iva",
    "KAR_VIS": "valor_base",
    "SUBTOTAL": "subtotal",
    "VRIVA": "valor_iva",
    "VRTOTAL": "total",
    "GRU_DES": "categoria",
}
df_ventas = df_ventas.rename(columns=renombrar_ventas)
df_ventas.head()

,factura,fecha,vehiculo,codigo,descripcion,cantidad,precio_unitario,iva,valor_base,subtotal,valor_iva,total,categoria
0,24145,2025-01-02,NaN,729,ACEITE MOBIL 20W50 4T,1.00,26000.00,0,0.00,26000.00,0.00,26000.00,ACEITES
1,24146,2025-01-02,NaN,4238,AMBIENTADOR LIGHT GEL,1.00,11764.70,19,2235.29,11764.70,2235.29,13999.99,ACCESORIOS
2,24147,2025-01-02,HYT526,517,MONTAJE DE LLANTAS,1.00,12605.00,19,2394.95,12605.00,2394.95,14999.95,SERVICIOS
3,24148,2025-01-02,LTS797,542,LAVADO EXTERIOR,1.00,27731.00,19,5268.89,27731.00,5268.89,32999.89,SERVICIOS
4,24148,2025-01-02,LTS797,547,LAVADO DE CHASIS,1.00,43697.00,19,8302.43,43697.00,8302.43,51999.43,SERVICIOS


In [16]:
columnas_texto = ["FAC_VEH", "REF_DES", "GRU_DES"]
nulos_antes = df_ventas_original.isna().sum()
duplicados_antes = int(df_ventas_original.duplicated().sum())
espacios_antes = df_ventas_original[columnas_texto].apply(
    lambda columna: columna.astype("string").str.contains(r"^\s+|\s+$", regex=True, na=False).sum()
)

print(f"Filas originales: {len(df_ventas_original)}")
print(f"Registros duplicados: {duplicados_antes}")
print("Valores nulos por columna:")
print(nulos_antes)
print("Filas con espacios al inicio o al final:")
print(espacios_antes)

Filas originales: 128086
Registros duplicados: 246
Valores nulos por columna:
FAC_NUM         0
FAC_FEC         0
FAC_VEH     29013
KAR_COD         0
REF_DES         0
KAR_SAL         0
KAR_PRE         0
KAR_IVA         0
KAR_VIS         0
SUBTOTAL        0
VRIVA           0
VRTOTAL         0
GRU_DES         0
dtype: int64
Filas con espacios al inicio o al final:
FAC_VEH        0
REF_DES    13827
GRU_DES        0
dtype: int64


In [17]:
df_ventas_limpio = df_ventas.copy()

for columna in ["descripcion", "categoria", "vehiculo"]:
    df_ventas_limpio[columna] = df_ventas_limpio[columna].astype("string").str.strip()

df_ventas_limpio["vehiculo"] = df_ventas_limpio["vehiculo"].fillna("SIN_PLACA")
df_ventas_limpio = df_ventas_limpio.drop_duplicates().reset_index(drop=True)

archivo_limpio = root / "data" / "processed" / "ventas_limpia.csv"
archivo_limpio.parent.mkdir(parents=True, exist_ok=True)
df_ventas_limpio.to_csv(archivo_limpio, index=False)

print(f"Archivo guardado: {archivo_limpio}")
df_ventas_limpio.head()

Archivo guardado: c:\Users\aleja\Proyecto-2\data\processed\ventas_limpia.csv


,factura,fecha,vehiculo,codigo,descripcion,cantidad,precio_unitario,iva,valor_base,subtotal,valor_iva,total,categoria
0,24145,2025-01-02,SIN_PLACA,729,ACEITE MOBIL 20W50 4T,1.00,26000.00,0,0.00,26000.00,0.00,26000.00,ACEITES
1,24146,2025-01-02,SIN_PLACA,4238,AMBIENTADOR LIGHT GEL,1.00,11764.70,19,2235.29,11764.70,2235.29,13999.99,ACCESORIOS
2,24147,2025-01-02,HYT526,517,MONTAJE DE LLANTAS,1.00,12605.00,19,2394.95,12605.00,2394.95,14999.95,SERVICIOS
3,24148,2025-01-02,LTS797,542,LAVADO EXTERIOR,1.00,27731.00,19,5268.89,27731.00,5268.89,32999.89,SERVICIOS
4,24148,2025-01-02,LTS797,547,LAVADO DE CHASIS,1.00,43697.00,19,8302.43,43697.00,8302.43,51999.43,SERVICIOS


La comparación muestra cuántas filas y duplicados tenía el archivo original y cuántos quedan después de limpiar. Las ventas sin placa se conservan con el texto `SIN_PLACA`; no se eliminan por ese motivo.

In [18]:
espacios_despues = int(
    df_ventas_limpio["descripcion"].astype("string").str.contains(r"^\s+|\s+$", regex=True, na=False).sum()
)
comparacion_ventas = pd.DataFrame({
    "Métrica": ["Filas", "Duplicados", "Vehículos sin placa", "Descripciones con espacios externos"],
    "Antes": [
        len(df_ventas_original),
        duplicados_antes,
        int(df_ventas_original["FAC_VEH"].isna().sum()),
        int(df_ventas_original["REF_DES"].astype("string").str.contains(r"^\s+|\s+$", regex=True, na=False).sum()),
    ],
    "Después": [
        len(df_ventas_limpio),
        int(df_ventas_limpio.duplicated().sum()),
        int((df_ventas_limpio["vehiculo"] == "SIN_PLACA").sum()),
        espacios_despues,
    ],
})
display(comparacion_ventas)

,Métrica,Antes,Después
0,Filas,128086,127840
1,Duplicados,246,0
2,Vehículos sin placa,29013,28897
3,Descripciones con espacios externos,13827,0
